In [18]:
import os
import json
import random
from pathlib import Path
from typing import Dict, List

import numpy as np

In [19]:
def collect_frame_embeddings(folder: str) -> List[str]:
    return sorted(str(p) for p in Path(folder).rglob("*_frame.npy"))

In [20]:
def load_embedding(path: str) -> np.ndarray:
    x = np.load(path)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D embedding (T, D), got shape {x.shape} for {path}")

    return x.astype(np.float32)

In [21]:
def cosine_sim(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + eps)
    return np.sum(a_n * b_n, axis=1)


def local_diff_energy(x: np.ndarray) -> float:
    if len(x) < 2:
        return 0.0
    diff = x[1:] - x[:-1]
    return float(np.mean(np.sum(diff * diff, axis=1)))


def adjacent_cosine(x: np.ndarray, eps: float = 1e-8) -> float:
    if len(x) < 2:
        return 1.0
    return float(np.mean(cosine_sim(x[:-1], x[1:], eps=eps)))


def moving_average_1d(x: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    if kernel_size <= 1 or len(x) < 2:
        return x.copy()

    pad = kernel_size // 2
    x_pad = np.pad(x, ((pad, pad), (0, 0)), mode="edge")
    out = np.zeros_like(x)

    for t in range(len(x)):
        out[t] = np.mean(x_pad[t:t + kernel_size], axis=0)

    return out

In [22]:
def estimate_articulation_stats_unpaired(
    clean_files: List[str],
    moderate_files: List[str],
    max_clean: int = 100,
    max_moderate: int = 100,
    seed: int = 42
) -> Dict:
    clean_files = [p for p in clean_files if p.endswith("_frame.npy")]
    moderate_files = [p for p in moderate_files if p.endswith("_frame.npy")]

    rng = random.Random(seed)

    if len(clean_files) == 0:
        raise ValueError("No clean frame files found.")
    if len(moderate_files) == 0:
        raise ValueError("No moderate frame files found.")

    clean_subset = rng.sample(clean_files, min(len(clean_files), max_clean))
    moderate_subset = rng.sample(moderate_files, min(len(moderate_files), max_moderate))

    clean_local_energy = []
    moderate_local_energy = []
    clean_adj_cos = []
    moderate_adj_cos = []
    clean_residual_l2 = []
    moderate_residual_l2 = []
    clean_frame_norms = []
    moderate_frame_norms = []

    for path in clean_subset:
        x = load_embedding(path)
        clean_local_energy.append(local_diff_energy(x))
        clean_adj_cos.append(adjacent_cosine(x))

        x_ma = moving_average_1d(x, kernel_size=3)
        residual = x - x_ma
        clean_residual_l2.extend(np.linalg.norm(residual, axis=1).tolist())
        clean_frame_norms.extend(np.linalg.norm(x, axis=1).tolist())

    for path in moderate_subset:
        x = load_embedding(path)
        moderate_local_energy.append(local_diff_energy(x))
        moderate_adj_cos.append(adjacent_cosine(x))

        x_ma = moving_average_1d(x, kernel_size=3)
        residual = x - x_ma
        moderate_residual_l2.extend(np.linalg.norm(residual, axis=1).tolist())
        moderate_frame_norms.extend(np.linalg.norm(x, axis=1).tolist())

    clean_residual_l2 = np.asarray(clean_residual_l2, dtype=np.float32)
    moderate_residual_l2 = np.asarray(moderate_residual_l2, dtype=np.float32)
    clean_frame_norms = np.asarray(clean_frame_norms, dtype=np.float32)
    moderate_frame_norms = np.asarray(moderate_frame_norms, dtype=np.float32)

    stats = {
        "num_clean_files_used": len(clean_subset),
        "num_moderate_files_used": len(moderate_subset),

        "clean_local_energy_mean": float(np.mean(clean_local_energy)),
        "moderate_local_energy_mean": float(np.mean(moderate_local_energy)),
        "smoothness_ratio": float(np.mean(moderate_local_energy) / (np.mean(clean_local_energy) + 1e-8)),

        "clean_adjacent_cos_mean": float(np.mean(clean_adj_cos)),
        "moderate_adjacent_cos_mean": float(np.mean(moderate_adj_cos)),
        "boundary_blur_gain": float(np.mean(moderate_adj_cos) - np.mean(clean_adj_cos)),

        "clean_residual_l2_mean": float(np.mean(clean_residual_l2)),
        "moderate_residual_l2_mean": float(np.mean(moderate_residual_l2)),
        "drift_l2_mean": float(np.mean(moderate_residual_l2)),
        "drift_l2_std": float(np.std(moderate_residual_l2)),
        "drift_l2_p95": float(np.percentile(moderate_residual_l2, 95)),

        "clean_frame_norm_mean": float(np.mean(clean_frame_norms)),
        "moderate_frame_norm_mean": float(np.mean(moderate_frame_norms)),
    }

    return stats

In [36]:
def apply_articulation_corruption_drift_only(
    x_clean: np.ndarray,
    stats: Dict,
    strength: float = 0.5,
    noise_smooth_kernel: int = 5,
    seed: int = 42
):
    """
    Drift-only articulation corruption.

    Rationale:
    Your measured stats do not support extra smoothing or boundary blur.
    So articulation is modeled as structured low-frequency perturbation only.
    """
    rng = np.random.default_rng(seed)

    x = x_clean.astype(np.float32).copy()
    T, D = x.shape

    drift_mean = stats.get("drift_l2_mean", 1.0)
    drift_std = stats.get("drift_l2_std", 0.5)
    drift_p95 = stats.get("drift_l2_p95", drift_mean + 2 * drift_std)

    # low-frequency random direction over time
    raw_noise = rng.normal(0.0, 1.0, size=(T, D)).astype(np.float32)
    smooth_noise = moving_average_1d(raw_noise, kernel_size=noise_smooth_kernel)

    norms = np.linalg.norm(smooth_noise, axis=1, keepdims=True) + 1e-8
    direction = smooth_noise / norms

    # framewise perturbation magnitude grounded in moderate residual stats
    mag = rng.normal(
        loc=drift_mean,
        scale=max(drift_std, 1e-6),
        size=(T, 1)
    ).astype(np.float32)

    mag = np.clip(mag, 0.0, drift_p95)

    # conservative scaling so corruption remains mild
    mag *= 0.40 * strength

    x_corr = x + mag * direction

    meta = {
        "alpha_smooth": 0.0,
        "alpha_blur": 0.0,
        "drift_scale_mean_applied": float(np.mean(mag)),
        "noise_smooth_kernel": noise_smooth_kernel,
        "strength": float(strength),
    }

    return x_corr.astype(np.float32), meta

In [37]:
def seq_cosine_mean(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + eps)
    return float(np.mean(np.sum(a_n * b_n, axis=1)))

In [38]:
clean_root = r"D:\Masters\Speech\Embeddings\F_Con"
moderate_root = r"D:\Masters\Speech\Embeddings\moderate"

clean_files = collect_frame_embeddings(clean_root)
moderate_files = collect_frame_embeddings(moderate_root)

print("Clean frame files:", len(clean_files))
print("Moderate frame files:", len(moderate_files))
print("Sample clean:", clean_files[:3])

stats = estimate_articulation_stats_unpaired(
    clean_files,
    moderate_files,
    max_clean=100,
    max_moderate=100,
    seed=42
)

print(json.dumps(stats, indent=2))

Clean frame files: 2158
Moderate frame files: 1065
Sample clean: ['D:\\Masters\\Speech\\Embeddings\\F_Con\\wav_headMic_FC01S01\\wav_headMic_FC01S01_0001_frame.npy', 'D:\\Masters\\Speech\\Embeddings\\F_Con\\wav_headMic_FC01S01\\wav_headMic_FC01S01_0002_frame.npy', 'D:\\Masters\\Speech\\Embeddings\\F_Con\\wav_headMic_FC01S01\\wav_headMic_FC01S01_0003_frame.npy']
{
  "num_clean_files_used": 100,
  "num_moderate_files_used": 100,
  "clean_local_energy_mean": 20.050154579877855,
  "moderate_local_energy_mean": 20.260493206977845,
  "smoothness_ratio": 1.010490623209767,
  "clean_adjacent_cos_mean": 0.9332037127017975,
  "moderate_adjacent_cos_mean": 0.9327216243743897,
  "boundary_blur_gain": -0.0004820883274078147,
  "clean_residual_l2_mean": 1.598952054977417,
  "moderate_residual_l2_mean": 1.5734976530075073,
  "drift_l2_mean": 1.5734976530075073,
  "drift_l2_std": 1.719301462173462,
  "drift_l2_p95": 5.36539888381958,
  "clean_frame_norm_mean": 11.770925521850586,
  "moderate_frame_norm

In [39]:
x_clean = load_embedding(clean_files[0])

x_corr, meta = apply_articulation_corruption_drift_only(
    x_clean=x_clean,
    stats=stats,
    strength=0.7,
    noise_smooth_kernel=5,
    seed=42
)

print("x_clean shape:", x_clean.shape)
print("x_corr shape :", x_corr.shape)
print("meta:", meta)

print("clean local energy:", local_diff_energy(x_clean))
print("corr  local energy:", local_diff_energy(x_corr))
print("clean adjacent cos:", adjacent_cosine(x_clean))
print("corr  adjacent cos:", adjacent_cosine(x_corr))
print("clean-corr mean frame cosine:", seq_cosine_mean(x_clean, x_corr))

x_clean shape: (537, 1024)
x_corr shape : (537, 1024)
meta: {'alpha_smooth': 0.0, 'alpha_blur': 0.0, 'drift_scale_mean_applied': 0.47147810459136963, 'noise_smooth_kernel': 5, 'strength': 0.7}
clean local energy: 24.119604110717773
corr  local energy: 24.548240661621094
clean adjacent cos: 0.9202415347099304
corr  adjacent cos: 0.9189842343330383
clean-corr mean frame cosine: 0.9986485838890076


In [40]:
def corrupt_clean_folder_with_articulation_drift_only(
    clean_files: List[str],
    clean_root: str,
    out_root: str,
    stats: Dict,
    strength: float = 0.7,
    noise_smooth_kernel: int = 5,
    seed: int = 42
):
    for idx, clean_path in enumerate(clean_files, start=1):
        x_clean = load_embedding(clean_path)

        x_corr, meta = apply_articulation_corruption_drift_only(
            x_clean=x_clean,
            stats=stats,
            strength=strength,
            noise_smooth_kernel=noise_smooth_kernel,
            seed=seed + idx
        )

        rel = str(Path(clean_path).resolve().relative_to(Path(clean_root).resolve()))
        out_path = os.path.join(out_root, rel)

        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        np.save(out_path, x_corr)

        print(
            f"[{idx:04d}/{len(clean_files):04d}] {Path(clean_path).name} | "
            f"drift={meta['drift_scale_mean_applied']:.4f}"
        )

In [41]:
out_root = r"D:\Masters\Speech\corrupted_clean_articulation_drift_only_v2"

corrupt_clean_folder_with_articulation_drift_only(
    clean_files=clean_files,
    clean_root=clean_root,
    out_root=out_root,
    stats=stats,
    strength=0.7,
    noise_smooth_kernel=5,
    seed=42
)

[0001/2158] wav_headMic_FC01S01_0001_frame.npy | drift=0.4691
[0002/2158] wav_headMic_FC01S01_0002_frame.npy | drift=0.4939
[0003/2158] wav_headMic_FC01S01_0003_frame.npy | drift=0.4857
[0004/2158] wav_headMic_FC01S01_0004_frame.npy | drift=0.4787
[0005/2158] wav_headMic_FC01S01_0005_frame.npy | drift=0.5140
[0006/2158] wav_headMic_FC01S01_0006_frame.npy | drift=0.4608
[0007/2158] wav_headMic_FC01S01_0007_frame.npy | drift=0.4946
[0008/2158] wav_headMic_FC01S01_0008_frame.npy | drift=0.4631
[0009/2158] wav_headMic_FC01S01_0009_frame.npy | drift=0.5427
[0010/2158] wav_headMic_FC01S01_0010_frame.npy | drift=0.5689
[0011/2158] wav_headMic_FC01S01_0011_frame.npy | drift=0.5119
[0012/2158] wav_headMic_FC01S01_0012_frame.npy | drift=0.5061
[0013/2158] wav_headMic_FC01S01_0013_frame.npy | drift=0.5666
[0014/2158] wav_headMic_FC01S01_0014_frame.npy | drift=0.4695
[0015/2158] wav_headMic_FC01S01_0015_frame.npy | drift=0.4817
[0016/2158] wav_headMic_FC01S01_0016_frame.npy | drift=0.4835
[0017/21

Diff Process

In [42]:
import os
import json
import math
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [43]:
def collect_frame_embeddings(folder: str) -> List[str]:
    return sorted(str(p) for p in Path(folder).rglob("*_frame.npy"))


def load_embedding(path: str) -> np.ndarray:
    x = np.load(path)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D embedding (T, D), got shape {x.shape} for {path}")
    return x.astype(np.float32)


def save_npy(arr: np.ndarray, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, arr)


def save_json(obj: Dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

In [44]:
def cosine_sim(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + eps)
    return np.sum(a_n * b_n, axis=1)


def seq_cosine_mean(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + eps)
    return float(np.mean(np.sum(a_n * b_n, axis=1)))


def local_diff_energy(x: np.ndarray) -> float:
    if len(x) < 2:
        return 0.0
    diff = x[1:] - x[:-1]
    return float(np.mean(np.sum(diff * diff, axis=1)))


def adjacent_cosine(x: np.ndarray, eps: float = 1e-8) -> float:
    if len(x) < 2:
        return 1.0
    return float(np.mean(cosine_sim(x[:-1], x[1:], eps=eps)))


def moving_average_1d(x: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    if kernel_size <= 1 or len(x) < 2:
        return x.copy()

    pad = kernel_size // 2
    x_pad = np.pad(x, ((pad, pad), (0, 0)), mode="edge")
    out = np.zeros_like(x)

    for t in range(len(x)):
        out[t] = np.mean(x_pad[t:t + kernel_size], axis=0)

    return out

In [45]:
def estimate_articulation_stats_unpaired(
    clean_files: List[str],
    moderate_files: List[str],
    max_clean: int = 100,
    max_moderate: int = 100,
    seed: int = 42
) -> Dict:
    clean_files = [p for p in clean_files if p.endswith("_frame.npy")]
    moderate_files = [p for p in moderate_files if p.endswith("_frame.npy")]

    rng = random.Random(seed)

    if len(clean_files) == 0:
        raise ValueError("No clean frame files found.")
    if len(moderate_files) == 0:
        raise ValueError("No moderate frame files found.")

    clean_subset = rng.sample(clean_files, min(len(clean_files), max_clean))
    moderate_subset = rng.sample(moderate_files, min(len(moderate_files), max_moderate))

    clean_local_energy = []
    moderate_local_energy = []
    clean_adj_cos = []
    moderate_adj_cos = []
    clean_residual_l2 = []
    moderate_residual_l2 = []
    clean_frame_norms = []
    moderate_frame_norms = []

    for path in clean_subset:
        x = load_embedding(path)
        clean_local_energy.append(local_diff_energy(x))
        clean_adj_cos.append(adjacent_cosine(x))

        x_ma = moving_average_1d(x, kernel_size=3)
        residual = x - x_ma
        clean_residual_l2.extend(np.linalg.norm(residual, axis=1).tolist())
        clean_frame_norms.extend(np.linalg.norm(x, axis=1).tolist())

    for path in moderate_subset:
        x = load_embedding(path)
        moderate_local_energy.append(local_diff_energy(x))
        moderate_adj_cos.append(adjacent_cosine(x))

        x_ma = moving_average_1d(x, kernel_size=3)
        residual = x - x_ma
        moderate_residual_l2.extend(np.linalg.norm(residual, axis=1).tolist())
        moderate_frame_norms.extend(np.linalg.norm(x, axis=1).tolist())

    clean_residual_l2 = np.asarray(clean_residual_l2, dtype=np.float32)
    moderate_residual_l2 = np.asarray(moderate_residual_l2, dtype=np.float32)
    clean_frame_norms = np.asarray(clean_frame_norms, dtype=np.float32)
    moderate_frame_norms = np.asarray(moderate_frame_norms, dtype=np.float32)

    stats = {
        "num_clean_files_used": len(clean_subset),
        "num_moderate_files_used": len(moderate_subset),

        "clean_local_energy_mean": float(np.mean(clean_local_energy)),
        "moderate_local_energy_mean": float(np.mean(moderate_local_energy)),
        "smoothness_ratio": float(np.mean(moderate_local_energy) / (np.mean(clean_local_energy) + 1e-8)),

        "clean_adjacent_cos_mean": float(np.mean(clean_adj_cos)),
        "moderate_adjacent_cos_mean": float(np.mean(moderate_adj_cos)),
        "boundary_blur_gain": float(np.mean(moderate_adj_cos) - np.mean(clean_adj_cos)),

        "clean_residual_l2_mean": float(np.mean(clean_residual_l2)),
        "moderate_residual_l2_mean": float(np.mean(moderate_residual_l2)),
        "drift_l2_mean": float(np.mean(moderate_residual_l2)),
        "drift_l2_std": float(np.std(moderate_residual_l2)),
        "drift_l2_p95": float(np.percentile(moderate_residual_l2, 95)),

        "clean_frame_norm_mean": float(np.mean(clean_frame_norms)),
        "moderate_frame_norm_mean": float(np.mean(moderate_frame_norms)),
    }

    return stats

In [46]:
def get_diffusion_strength(t: int, T: int, mode: str = "cosine") -> float:
    if T <= 1:
        return 1.0

    frac = t / T

    if mode == "linear":
        return frac
    elif mode == "cosine":
        return 0.5 * (1.0 - np.cos(np.pi * frac))
    else:
        raise ValueError(f"Unsupported schedule: {mode}")

In [57]:
def get_step_weights(T: int, schedule: str = "cosine") -> np.ndarray:
    weights = []
    for t in range(1, T + 1):
        s = get_diffusion_strength(t, T, mode=schedule)
        weights.append(s)

    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / (weights.sum() + 1e-8)
    return weights

In [ ]:
def articulation_forward_step_budgeted(
    x_prev: np.ndarray,
    stats: dict,
    step_weight: float,
    total_scale: float = 0.60,
    noise_smooth_kernel: int = 7,
    embed_smooth_kernel: int = 3,
    max_blur_alpha: float = 0.35,
    max_mix_prob: float = 0.25,
    max_mix_lam: float = 0.35,
    seed: int = 42
):
    """
    One forward articulation step with normalized budget.
    Adds:
      1. smooth additive drift
      2. embedding smoothing (boundary blur)
      3. local frame mixing (slurring)
    """
    rng = np.random.default_rng(seed)

    x_prev = x_prev.astype(np.float32)
    Tseq, D = x_prev.shape

    drift_mean = stats.get("drift_l2_mean", 1.0)
    drift_std = stats.get("drift_l2_std", 0.5)
    drift_p95 = stats.get("drift_l2_p95", drift_mean + 2 * drift_std)

    # --- smooth directional noise ---
    raw_noise = rng.normal(0.0, 1.0, size=(Tseq, D)).astype(np.float32)
    smooth_noise = moving_average_1d(raw_noise, kernel_size=noise_smooth_kernel)

    norms = np.linalg.norm(smooth_noise, axis=1, keepdims=True) + 1e-8
    direction = smooth_noise / norms

    mag = rng.normal(
        loc=drift_mean,
        scale=max(drift_std, 1e-6),
        size=(Tseq, 1)
    ).astype(np.float32)
    mag = np.clip(mag, 0.0, drift_p95)

    # stronger per-step corruption
    mag *= total_scale * step_weight

    delta_t = mag * direction
    x_t = x_prev + delta_t

    # --- embedding smoothing / boundary blur ---
    blur_alpha = max_blur_alpha * step_weight
    if blur_alpha > 0:
        x_smooth = moving_average_1d(x_t, kernel_size=embed_smooth_kernel)
        x_t = (1.0 - blur_alpha) * x_t + blur_alpha * x_smooth

    # --- neighbor frame mixing / slurring ---
    mix_prob = max_mix_prob * step_weight
    mix_lam = max_mix_lam * step_weight
    if mix_prob > 0 and mix_lam > 0:
        shift = rng.choice([-1, 1])
        x_shift = np.roll(x_prev, shift=shift, axis=0)
        mask = (rng.random((Tseq, 1)) < mix_prob).astype(np.float32)
        x_t = (1.0 - mask) * x_t + mask * ((1.0 - mix_lam) * x_t + mix_lam * x_shift)

    # --- renormalize to preserve approximate corruption budget ---
    delta_t = x_t - x_prev
    delta_norm = np.linalg.norm(delta_t, axis=1, keepdims=True) + 1e-8
    delta_t = (delta_t / delta_norm) * mag
    x_t = x_prev + delta_t

    meta = {
        "step_weight": float(step_weight),
        "delta_mean_applied": float(np.mean(mag)),
        "noise_smooth_kernel": noise_smooth_kernel,
        "embed_smooth_kernel": embed_smooth_kernel,
        "blur_alpha": float(blur_alpha),
        "mix_prob": float(mix_prob),
        "mix_lam": float(mix_lam),
        "total_scale": float(total_scale),
    }

    return x_t.astype(np.float32), delta_t.astype(np.float32), meta

In [ ]:
def articulation_forward_diffusion_chain_budgeted(
    x_clean: np.ndarray,
    stats: dict,
    T: int = 30,
    total_scale: float = 0.30,
    noise_smooth_kernel: int = 5,
    schedule: str = "cosine",
    seed: int = 42
):
    """
    Forward chain with normalized total corruption budget.
    """
    x_states = [x_clean.astype(np.float32).copy()]
    deltas = [None]
    metas = [None]

    x = x_clean.astype(np.float32).copy()
    step_weights = get_step_weights(T, schedule=schedule)

    for t in range(1, T + 1):
        x, delta_t, meta_t = articulation_forward_step_budgeted(
            x_prev=x,
            stats=stats,
            step_weight=float(step_weights[t - 1]),
            total_scale=total_scale,
            noise_smooth_kernel=noise_smooth_kernel,
            seed=seed + t
        )
        meta_t["t"] = t
        x_states.append(x)
        deltas.append(delta_t)
        metas.append(meta_t)

    return x_states, deltas, metas

In [49]:
def articulation_reverse_oracle_chain(
    x_T: np.ndarray,
    deltas: list
):
    T = len(deltas) - 1
    x_states_rev = [None] * (T + 1)
    x_states_rev[T] = x_T.astype(np.float32).copy()

    x = x_T.astype(np.float32).copy()

    for t in range(T, 0, -1):
        x = x - deltas[t]
        x_states_rev[t - 1] = x.astype(np.float32).copy()

    return x_states_rev

In [59]:
clean_root = r"D:\Masters\Speech\Embeddings\F_Con"
moderate_root = r"D:\Masters\Speech\Embeddings\moderate"

clean_files = collect_frame_embeddings(clean_root)
moderate_files = collect_frame_embeddings(moderate_root)

print("Clean frame files:", len(clean_files))
print("Moderate frame files:", len(moderate_files))
print("Sample clean:", clean_files[:3])

stats = estimate_articulation_stats_unpaired(
    clean_files,
    moderate_files,
    max_clean=100,
    max_moderate=100,
    seed=42
)

print(json.dumps(stats, indent=2))

x_clean = load_embedding(clean_files[0])

Tdiff = 30

x_states, deltas, metas = articulation_forward_diffusion_chain_budgeted(
    x_clean=x_clean,
    stats=stats,
    T=Tdiff,
    total_scale=1.50,
    noise_smooth_kernel=5,
    schedule="cosine",
    seed=42
)

x_rev_states = articulation_reverse_oracle_chain(
    x_T=x_states[-1],
    deltas=deltas
)

x_T = x_states[-1]
x_0_recovered = x_rev_states[0]

print("x_clean shape:", x_clean.shape)
print("x_T shape:", x_T.shape)
print("x_0 recovered shape:", x_0_recovered.shape)

print("clean -> x_T cosine:", seq_cosine_mean(x_clean, x_T))
print("clean -> recovered cosine:", seq_cosine_mean(x_clean, x_0_recovered))
print("recovery mean abs error:", float(np.mean(np.abs(x_clean - x_0_recovered))))
print("recovery max abs error:", float(np.max(np.abs(x_clean - x_0_recovered))))

Clean frame files: 2158
Moderate frame files: 1065
Sample clean: ['D:\\Masters\\Speech\\Embeddings\\F_Con\\wav_headMic_FC01S01\\wav_headMic_FC01S01_0001_frame.npy', 'D:\\Masters\\Speech\\Embeddings\\F_Con\\wav_headMic_FC01S01\\wav_headMic_FC01S01_0002_frame.npy', 'D:\\Masters\\Speech\\Embeddings\\F_Con\\wav_headMic_FC01S01\\wav_headMic_FC01S01_0003_frame.npy']
{
  "num_clean_files_used": 100,
  "num_moderate_files_used": 100,
  "clean_local_energy_mean": 20.050154579877855,
  "moderate_local_energy_mean": 20.260493206977845,
  "smoothness_ratio": 1.010490623209767,
  "clean_adjacent_cos_mean": 0.9332037127017975,
  "moderate_adjacent_cos_mean": 0.9327216243743897,
  "boundary_blur_gain": -0.0004820883274078147,
  "clean_residual_l2_mean": 1.598952054977417,
  "moderate_residual_l2_mean": 1.5734976530075073,
  "drift_l2_mean": 1.5734976530075073,
  "drift_l2_std": 1.719301462173462,
  "drift_l2_p95": 5.36539888381958,
  "clean_frame_norm_mean": 11.770925521850586,
  "moderate_frame_norm

In [60]:
def save_articulation_chain_to_disk(
    clean_files: List[str],
    clean_root: str,
    out_root: str,
    stats: Dict,
    T: int = 30,
    base_scale: float = 0.30,
    noise_smooth_kernel: int = 5,
    schedule: str = "cosine",
    seed: int = 42
):
    """
    Saves:
      out_root/
        states/<relative_path_without_ext>/x_000.npy ... x_030.npy
        deltas/<relative_path_without_ext>/delta_001.npy ... delta_030.npy
        meta/<relative_path_without_ext>.json
    """
    for idx, clean_path in enumerate(clean_files, start=1):
        x_clean = load_embedding(clean_path)

        x_states, deltas, metas = articulation_forward_diffusion_chain_budgeted(
            x_clean=x_clean,
            stats=stats,
            T=T,
            total_scale=base_scale,
            noise_smooth_kernel=noise_smooth_kernel,
            schedule=schedule,
            seed=seed + idx
        )

        rel = str(Path(clean_path).resolve().relative_to(Path(clean_root).resolve()))
        rel_no_ext = str(Path(rel).with_suffix(""))

        state_dir = os.path.join(out_root, "states", rel_no_ext)
        delta_dir = os.path.join(out_root, "deltas", rel_no_ext)
        meta_path = os.path.join(out_root, "meta", rel_no_ext + ".json")

        os.makedirs(state_dir, exist_ok=True)
        os.makedirs(delta_dir, exist_ok=True)

        for t in range(T + 1):
            save_npy(x_states[t], os.path.join(state_dir, f"x_{t:03d}.npy"))

        for t in range(1, T + 1):
            save_npy(deltas[t], os.path.join(delta_dir, f"delta_{t:03d}.npy"))

        meta_obj = {
            "source_clean_file": clean_path,
            "T": T,
            "base_scale": base_scale,
            "noise_smooth_kernel": noise_smooth_kernel,
            "schedule": schedule,
            "seed_used": int(seed + idx),
            "step_meta": metas[1:],
        }
        save_json(meta_obj, meta_path)

        print(
            f"[{idx:04d}/{len(clean_files):04d}] {Path(clean_path).name} | "
            f"xT_cos={seq_cosine_mean(x_clean, x_states[-1]):.6f}"
        )

In [61]:
art_chain_root = r"D:\Masters\Speech\articulation_diffusion_chain"

save_articulation_chain_to_disk(
    clean_files=clean_files,
    clean_root=clean_root,
    out_root=art_chain_root,
    stats=stats,
    T=30,
    base_scale=1.5,
    noise_smooth_kernel=5,
    schedule="cosine",
    seed=42
)

[0001/2158] wav_headMic_FC01S01_0001_frame.npy | xT_cos=0.998062
[0002/2158] wav_headMic_FC01S01_0002_frame.npy | xT_cos=0.998022
[0003/2158] wav_headMic_FC01S01_0003_frame.npy | xT_cos=0.997451
[0004/2158] wav_headMic_FC01S01_0004_frame.npy | xT_cos=0.997987
[0005/2158] wav_headMic_FC01S01_0005_frame.npy | xT_cos=0.997911
[0006/2158] wav_headMic_FC01S01_0006_frame.npy | xT_cos=0.997853
[0007/2158] wav_headMic_FC01S01_0007_frame.npy | xT_cos=0.997731
[0008/2158] wav_headMic_FC01S01_0008_frame.npy | xT_cos=0.997971
[0009/2158] wav_headMic_FC01S01_0009_frame.npy | xT_cos=0.997750
[0010/2158] wav_headMic_FC01S01_0010_frame.npy | xT_cos=0.997950
[0011/2158] wav_headMic_FC01S01_0011_frame.npy | xT_cos=0.997902
[0012/2158] wav_headMic_FC01S01_0012_frame.npy | xT_cos=0.998000
[0013/2158] wav_headMic_FC01S01_0013_frame.npy | xT_cos=0.997625
[0014/2158] wav_headMic_FC01S01_0014_frame.npy | xT_cos=0.997802
[0015/2158] wav_headMic_FC01S01_0015_frame.npy | xT_cos=0.998166
[0016/2158] wav_headMic_F

In [62]:
class ArticulationReverseDataset(Dataset):
    def __init__(self, chain_root: str):
        self.items = []

        states_root = Path(chain_root) / "states"
        deltas_root = Path(chain_root) / "deltas"

        state_dirs = sorted([p for p in states_root.rglob("*") if p.is_dir()])

        for state_dir in state_dirs:
            rel = state_dir.relative_to(states_root)
            delta_dir = deltas_root / rel

            x_files = sorted(state_dir.glob("x_*.npy"))
            delta_files = sorted(delta_dir.glob("delta_*.npy"))

            if len(x_files) < 2 or len(delta_files) < 1:
                continue

            T = len(delta_files)

            for t in range(1, T + 1):
                x_t_path = state_dir / f"x_{t:03d}.npy"
                x_prev_path = state_dir / f"x_{t-1:03d}.npy"
                delta_t_path = delta_dir / f"delta_{t:03d}.npy"

                if x_t_path.exists() and x_prev_path.exists() and delta_t_path.exists():
                    self.items.append({
                        "x_t": str(x_t_path),
                        "x_prev": str(x_prev_path),
                        "delta_t": str(delta_t_path),
                        "t": t,
                        "T": T,
                    })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        x_t = np.load(item["x_t"]).astype(np.float32)
        x_prev = np.load(item["x_prev"]).astype(np.float32)
        delta_t = np.load(item["delta_t"]).astype(np.float32)

        return {
            "x_t": torch.from_numpy(x_t),         # (L, D)
            "x_prev": torch.from_numpy(x_prev),   # (L, D)
            "delta_t": torch.from_numpy(delta_t), # (L, D)
            "t": torch.tensor(item["t"], dtype=torch.long),
            "T": torch.tensor(item["T"], dtype=torch.long),
            "length": torch.tensor(x_t.shape[0], dtype=torch.long),
        }


def articulation_collate_fn(batch):
    lengths = [b["length"].item() for b in batch]
    max_len = max(lengths)
    D = batch[0]["x_t"].shape[1]

    B = len(batch)

    x_t_pad = torch.zeros(B, max_len, D, dtype=torch.float32)
    x_prev_pad = torch.zeros(B, max_len, D, dtype=torch.float32)
    delta_pad = torch.zeros(B, max_len, D, dtype=torch.float32)
    mask = torch.zeros(B, max_len, dtype=torch.bool)

    t_vals = []
    T_vals = []

    for i, b in enumerate(batch):
        L = b["length"].item()
        x_t_pad[i, :L] = b["x_t"]
        x_prev_pad[i, :L] = b["x_prev"]
        delta_pad[i, :L] = b["delta_t"]
        mask[i, :L] = True
        t_vals.append(b["t"])
        T_vals.append(b["T"])

    t_vals = torch.stack(t_vals, dim=0)
    T_vals = torch.stack(T_vals, dim=0)

    return {
        "x_t": x_t_pad,
        "x_prev": x_prev_pad,
        "delta_t": delta_pad,
        "mask": mask,
        "t": t_vals,
        "T": T_vals,
        "lengths": torch.tensor(lengths, dtype=torch.long),
    }

In [63]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t_norm: torch.Tensor) -> torch.Tensor:
        """
        t_norm: (B,) in [0,1]
        returns: (B, dim)
        """
        device = t_norm.device
        half = self.dim // 2

        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=device, dtype=torch.float32) / max(half - 1, 1)
        )

        args = t_norm[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))

        return emb

In [64]:
class ReverseArticulationStepModel(nn.Module):
    def __init__(
        self,
        input_dim: int = 1024,
        hidden_dim: int = 512,
        time_dim: int = 128,
        num_layers: int = 4,
        kernel_size: int = 5
    ):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.in_proj = nn.Linear(input_dim, hidden_dim)

        convs = []
        for _ in range(num_layers):
            convs.append(
                nn.Sequential(
                    nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=kernel_size // 2),
                    nn.SiLU(),
                    nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=kernel_size // 2),
                )
            )
        self.convs = nn.ModuleList(convs)

        self.out_proj = nn.Linear(hidden_dim, input_dim)

    def forward(self, x_t: torch.Tensor, t: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
        """
        x_t: (B, L, D)
        t:   (B,)
        T:   (B,)
        returns predicted delta_t: (B, L, D)
        """
        B, L, D = x_t.shape

        t_norm = t.float() / T.float().clamp(min=1.0)
        t_emb = self.time_embed(t_norm)               # (B, time_dim)
        t_feat = self.time_proj(t_emb)                # (B, hidden_dim)

        h = self.in_proj(x_t)                         # (B, L, hidden_dim)
        h = h + t_feat[:, None, :]                    # inject timestep

        h = h.transpose(1, 2)                         # (B, hidden_dim, L)

        for block in self.convs:
            residual = h
            h = block(h) + residual

        h = h.transpose(1, 2)                         # (B, L, hidden_dim)
        delta_hat = self.out_proj(h)                  # (B, L, D)

        return delta_hat

In [65]:
def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    pred, target: (B, L, D)
    mask: (B, L)
    """
    mask_f = mask.unsqueeze(-1).float()
    loss = ((pred - target) ** 2) * mask_f
    denom = mask_f.sum() * pred.shape[-1]
    return loss.sum() / denom.clamp(min=1.0)


def reverse_step_loss(batch, model, device):
    x_t = batch["x_t"].to(device)
    x_prev = batch["x_prev"].to(device)
    delta_t = batch["delta_t"].to(device)
    mask = batch["mask"].to(device)
    t = batch["t"].to(device)
    T = batch["T"].to(device)

    delta_hat = model(x_t, t, T)
    x_prev_hat = x_t - delta_hat

    loss_delta = masked_mse(delta_hat, delta_t, mask)
    loss_prev = masked_mse(x_prev_hat, x_prev, mask)

    loss = loss_delta + loss_prev

    return loss, {
        "loss": loss.item(),
        "loss_delta": loss_delta.item(),
        "loss_prev": loss_prev.item(),
    }

In [66]:
from torch.utils.data import random_split

chain_root = art_chain_root  # from previous save step

full_dataset = ArticulationReverseDataset(chain_root)
print("Total reverse-step training items:", len(full_dataset))

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=articulation_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=articulation_collate_fn
)

print("Train items:", len(train_dataset))
print("Val items:", len(val_dataset))

Total reverse-step training items: 64740
Train items: 58266
Val items: 6474


In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = ReverseArticulationStepModel(
    input_dim=1024,
    hidden_dim=512,
    time_dim=128,
    num_layers=4,
    kernel_size=5
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def evaluate_reverse_model(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            loss, stats_dict = reverse_step_loss(batch, model, device)
            total_loss += stats_dict["loss"]
            total_batches += 1

    return total_loss / max(total_batches, 1)


num_epochs = 10
best_val = float("inf")
best_ckpt_path = r"D:\Masters\Speech\reverse_articulation_step_model_best.pt"

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    running_delta = 0.0
    running_prev = 0.0
    num_batches = 0

    for batch in train_loader:
        optimizer.zero_grad()

        loss, stats_dict = reverse_step_loss(batch, model, device)
        loss.backward()
        optimizer.step()

        running_loss += stats_dict["loss"]
        running_delta += stats_dict["loss_delta"]
        running_prev += stats_dict["loss_prev"]
        num_batches += 1

    train_loss = running_loss / max(num_batches, 1)
    train_delta = running_delta / max(num_batches, 1)
    train_prev = running_prev / max(num_batches, 1)

    val_loss = evaluate_reverse_model(model, val_loader, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train={train_loss:.6f} | delta={train_delta:.6f} | prev={train_prev:.6f} | "
        f"val={val_loss:.6f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), best_ckpt_path)
        print(f"  Saved best model to {best_ckpt_path}")

Using device: cpu
Epoch 01 | train=0.000116 | delta=0.000058 | prev=0.000058 | val=0.000036
  Saved best model to D:\Masters\Speech\reverse_articulation_step_model_best.pt
Epoch 02 | train=0.000036 | delta=0.000018 | prev=0.000018 | val=0.000035
  Saved best model to D:\Masters\Speech\reverse_articulation_step_model_best.pt
Epoch 03 | train=0.000036 | delta=0.000018 | prev=0.000018 | val=0.000035
  Saved best model to D:\Masters\Speech\reverse_articulation_step_model_best.pt
Epoch 04 | train=0.000036 | delta=0.000018 | prev=0.000018 | val=0.000035
  Saved best model to D:\Masters\Speech\reverse_articulation_step_model_best.pt
Epoch 05 | train=0.000035 | delta=0.000018 | prev=0.000018 | val=0.000035
  Saved best model to D:\Masters\Speech\reverse_articulation_step_model_best.pt
Epoch 06 | train=0.000035 | delta=0.000018 | prev=0.000018 | val=0.000035
  Saved best model to D:\Masters\Speech\reverse_articulation_step_model_best.pt
Epoch 07 | train=0.000035 | delta=0.000018 | prev=0.000018

In [68]:
def run_learned_reverse_chain(
    model: nn.Module,
    x_T: np.ndarray,
    Tdiff: int,
    device: torch.device
):
    model.eval()

    x = torch.from_numpy(x_T.astype(np.float32)).unsqueeze(0).to(device)  # (1, L, D)
    states = [None] * (Tdiff + 1)
    states[Tdiff] = x_T.astype(np.float32).copy()

    with torch.no_grad():
        for t in range(Tdiff, 0, -1):
            t_tensor = torch.tensor([t], dtype=torch.long, device=device)
            T_tensor = torch.tensor([Tdiff], dtype=torch.long, device=device)

            delta_hat = model(x, t_tensor, T_tensor)
            x = x - delta_hat

            states[t - 1] = x.squeeze(0).cpu().numpy().astype(np.float32)

    return states

In [69]:
# load best model
model = ReverseArticulationStepModel(
    input_dim=1024,
    hidden_dim=512,
    time_dim=128,
    num_layers=4,
    kernel_size=5
).to(device)

model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
model.eval()

# use a clean file not necessarily the first one
test_clean = load_embedding(clean_files[10])

Tdiff = 30

# forward synthetic chain
x_states, deltas, metas = articulation_forward_diffusion_chain_budgeted(
    x_clean=test_clean,
    stats=stats,
    T=Tdiff,
    total_scale=1.50,
    noise_smooth_kernel=5,
    schedule="cosine",
    seed=999
)

x_T = x_states[-1]

# learned reverse chain
x_rev_learned = run_learned_reverse_chain(
    model=model,
    x_T=x_T,
    Tdiff=Tdiff,
    device=device
)

x_0_hat = x_rev_learned[0]

print("clean -> x_T cosine:", seq_cosine_mean(test_clean, x_T))
print("clean -> learned recovered cosine:", seq_cosine_mean(test_clean, x_0_hat))
print("learned reverse mean abs error:", float(np.mean(np.abs(test_clean - x_0_hat))))
print("learned reverse max abs error:", float(np.max(np.abs(test_clean - x_0_hat))))

clean -> x_T cosine: 0.9977728724479675
clean -> learned recovered cosine: 0.9986077547073364
learned reverse mean abs error: 0.014264869503676891
learned reverse max abs error: 0.11667882651090622


In [70]:
# load best model
model = ReverseArticulationStepModel(
    input_dim=1024,
    hidden_dim=512,
    time_dim=128,
    num_layers=4,
    kernel_size=5
).to(device)

model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
model.eval()

# use a clean file not necessarily the first one
test_clean = load_embedding(clean_files[10])

Tdiff = 30

# forward synthetic chain
x_states, deltas, metas = articulation_forward_diffusion_chain_budgeted(
    x_clean=test_clean,
    stats=stats,
    T=Tdiff,
    total_scale=1.50,
    noise_smooth_kernel=5,
    schedule="cosine",
    seed=999
)

x_T = x_states[-1]

# learned reverse chain
x_rev_learned = run_learned_reverse_chain(
    model=model,
    x_T=x_T,
    Tdiff=Tdiff,
    device=device
)

x_0_hat = x_rev_learned[0]

print("clean -> x_T cosine:", seq_cosine_mean(test_clean, x_T))
print("clean -> learned recovered cosine:", seq_cosine_mean(test_clean, x_0_hat))
print("learned reverse mean abs error:", float(np.mean(np.abs(test_clean - x_0_hat))))
print("learned reverse max abs error:", float(np.max(np.abs(test_clean - x_0_hat))))

clean -> x_T cosine: 0.9977728724479675
clean -> learned recovered cosine: 0.9986077547073364
learned reverse mean abs error: 0.014264869503676891
learned reverse max abs error: 0.11667882651090622


In [ ]:
save_learned_reverse_example(
    x_clean=test_clean,
    x_states=x_states,
    x_rev_states=x_rev_learned,
    out_dir=r"D:\Masters\Speech\articulation_reverse_example"
)